# Mongolian Intent BERT v3 - Training

GPT-augmented data (12,047 examples, 14 intents) дээр Mongolian BERT train хийнэ.

**Заавар:**
1. Runtime → Change runtime type → **T4 GPU** сонго
2. `training-data.json` файлаа upload хий (эхний cell ажиллуулахад асуухautomatically)
3. Бүх cell-ийг дараалуулан ажиллуул
4. Сүүлийн cell-ээс model файлуудыг татаж ав

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
# 2. Upload training-data.json
from google.colab import files
import json

print("training-data.json файлаа upload хийнэ үү...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
with open(filename) as f:
    raw = json.load(f)

from collections import Counter
counts = Counter(r['intent'] for r in raw)
print(f"\n✅ {len(raw)} examples loaded, {len(counts)} intents:")
for intent, count in counts.most_common():
    print(f"  {intent}: {count}")

In [ ]:
# 3. Prepare data
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# Filter intents with >= 10 examples
valid = {k for k, v in counts.items() if v >= 10}
data = [r for r in raw if r['intent'] in valid]
labels_sorted = sorted(valid)
label2id = {l: i for i, l in enumerate(labels_sorted)}
id2label = {i: l for l, i in label2id.items()}
NUM = len(labels_sorted)
print(f"{len(data)} examples, {NUM} intents")

texts = [r['text'] for r in data]
labels = [label2id[r['intent']] for r in data]
train_t, val_t, train_l, val_l = train_test_split(
    texts, labels, test_size=0.15, random_state=SEED, stratify=labels
)
print(f"Train: {len(train_t)}, Val: {len(val_t)}")

In [ ]:
# 4. Tokenize
from transformers import AutoTokenizer

MODEL_NAME = "tugstugi/bert-base-mongolian-cased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(ex):
    return tokenizer(ex['text'], padding='max_length', truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_dict({'text': train_t, 'label': train_l}).map(tok, batched=True, remove_columns=['text'])
val_ds = Dataset.from_dict({'text': val_t, 'label': val_l}).map(tok, batched=True, remove_columns=['text'])
train_ds.set_format('torch')
val_ds.set_format('torch')
print(f"✅ Tokenized. Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
# 5. Train BERT v3
from transformers import (
    AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM, id2label=id2label, label2id=label2id
)

def compute_metrics(p):
    return {'accuracy': accuracy_score(p.label_ids, np.argmax(p.predictions, axis=-1))}

EPOCHS = 10
BATCH_SIZE = 32  # Colab T4 can handle 32
LR = 3e-5

args = TrainingArguments(
    output_dir='./checkpoints-v3',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    fp16=True,  # GPU-тэй учир fp16 ашиглана
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"🚀 Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, fp16=True")
print(f"{'='*60}")
trainer.train()

In [ ]:
# 6. Evaluate
from sklearn.metrics import classification_report

result = trainer.evaluate()
print(f"\n✅ Val Accuracy: {result['eval_accuracy']:.4f}")

preds = np.argmax(trainer.predict(val_ds).predictions, axis=-1)
print("\nClassification Report:")
print(classification_report(val_l, preds, target_names=labels_sorted, digits=3))

In [ ]:
# 7. Save model
import os

save_path = './mongolian-intent-bert-v3'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

with open(os.path.join(save_path, 'label_map.json'), 'w') as f:
    json.dump({
        'label2id': label2id,
        'id2label': {str(k): v for k, v in id2label.items()}
    }, f, indent=2)

print(f"✅ Model saved to {save_path}")
print(f"Accuracy: {result['eval_accuracy']:.4f}")

In [ ]:
# 8. Download model files
import shutil

# Zip the model
shutil.make_archive('mongolian-intent-bert-v3', 'zip', save_path)
print("📦 Zipped model. Downloading...")

files.download('mongolian-intent-bert-v3.zip')

In [ ]:
# 9. (Optional) Quick test
from transformers import pipeline

clf = pipeline('text-classification', model=save_path, tokenizer=save_path, device=0)

test_msgs = [
    "сайн байна уу",
    "захиалга хаана явж байна",
    "буцааж болох уу",
    "баярлалаа",
    "төлбөрөө хийлээ",
    "XL размер байна уу",
    "хар өнгийн цамц хайж байна",
    "маш удаан хүргэлт",
    "глютенгүй юм бий юу",
    "ширээ захиалъя 4 хүн",
    "ямар бараа байна",
    "утас хэд вэ",
    "Улаанбаатар хот руу хүргэнэ үү",
    "zahialga hiih",
]

print("\n🧪 Quick test:")
print(f"{'Message':<40} {'Predicted':>20} {'Score':>8}")
print('-' * 70)
for msg in test_msgs:
    r = clf(msg)[0]
    print(f"{msg:<40} {r['label']:>20} {r['score']:>8.3f}")